In [9]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import time
from tqdm.notebook import tqdm
from google.cloud import storage

In [10]:
PROJECT = "creature-vision"
BUCKET_NAME = "creature-vision-training-set"
INPUT_PREFIXES = ["correct_predictions", "incorrect_predictions"]
OUTPUT_PREFIX = "v3_0/baseline"
MAX_PER_CLASS = 1
LABEL_SUFFIX = "_labels.json"
THREADS = 32

In [11]:
# Initialize GCS client
client = storage.Client(project=PROJECT)
bucket = client.bucket(BUCKET_NAME)

collected = defaultdict(list)
copy_tasks = []

def stream_sample_blobs(prefixes, max_per_class):
    collected = defaultdict(list)

    for prefix in prefixes:
        print(f"📁 Scanning: {prefix}", flush=True)
        for image_blob in bucket.list_blobs(prefix=prefix):  # Streamed
            if not image_blob.name.endswith(".jpg"):
                continue

            label_blob = bucket.blob(image_blob.name.replace(".jpg", LABEL_SUFFIX))
            if not label_blob.exists():
                continue

            try:
                label_data = json.loads(label_blob.download_as_text())
                class_name = label_data["api_label"]

                if len(collected[class_name]) >= max_per_class:
                    continue

                collected[class_name].append((image_blob, label_blob))

                if len(collected) >= 162 and all(len(v) >= max_per_class for v in collected.values()):
                    print("✅ Early stop: All classes filled", flush=True)
                    return collected

            except Exception as e:
                print(f"❌ Failed to parse label for {image_blob.name}: {e}", flush=True)

    print("⚠️ Finished scan without full sample set", flush=True)
    return collected


def copy_blob_pair(image_blob, label_blob):
    try:
        image_name = image_blob.name.split("/")[-1]
        label_name = label_blob.name.split("/")[-1]
        image_dest = f"{OUTPUT_PREFIX}/{image_name}"
        label_dest = f"{OUTPUT_PREFIX}/{label_name}"

        bucket.copy_blob(image_blob, bucket, image_dest)
        bucket.copy_blob(label_blob, bucket, label_dest)
        return True
    except Exception as e:
        print(f"❌ Error copying {image_blob.name}: {e}")
        return False

def main():
    print("🔍 Starting streamed sample collection...", flush=True)
    start = time.time()
    collected = stream_sample_blobs(INPUT_PREFIXES, MAX_PER_CLASS)
    print(f"⏱️ Sample collection complete in {time.time() - start:.2f}s", flush=True)

    total_samples = sum(len(v) for v in collected.values())
    print(f"📦 Preparing copy of {total_samples} samples...", flush=True)


# Parallel Copy
    copy_tasks = []
    with ThreadPoolExecutor(max_workers=THREADS) as executor:
        for pairs in collected.values():
            for image_blob, label_blob in pairs:
                copy_tasks.append(executor.submit(copy_blob_pair, image_blob, label_blob))

        for f in tqdm(as_completed(copy_tasks), total=len(copy_tasks), desc="Copying to GCS"):
            f.result()

    print("✅ Done. Class coverage:")
    for cls, samples in sorted(collected.items()):
        print(f"{cls}: {len(samples)}")

In [12]:
main()

🔍 Starting streamed sample collection...
📁 Scanning: correct_predictions


KeyboardInterrupt: 

In [ ]:

print("hello", flush=True)
for i in tqdm(range(5), desc="test loop"):
    time.sleep(1)


hello


test loop:   0%|          | 0/5 [00:00<?, ?it/s]